# Week 2 – Feature Engineering

## 2.1 Introduction

## 2.2 Create Time Features

## 2.3 Rolling Statistics

## 2.4 Lag Features

## 2.5 Growth Indicators

## 2.6 Feature Dataset

## 2.7 Week 2 Summary

## Objective

The objective of this notebook is to create additional features that improve the predictive capability of machine learning models. Time-series features such as lag values, rolling averages, year-over-year changes, and greenhouse gas intensity are generated.

### Importing Libraries

The required libraries are imported and the filtered dataset prepared in Week 1 is loaded.

### Loading the Filtered Dataset

The cleaned dataset generated during Week 1 is loaded for feature engineering.

In [18]:
import pandas as pd 
import numpy as np
filtered_dataset = pd.read_csv("data/filtered_dataset.csv")
filtered_dataset

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Australia,1990,AUS,17126304.0,4.641366e+11,3.463,0.202,278.061,21.223,8.263,...,0.666,1.693,0.007,0.008,0.017,0.002,538.479,336.927,-33.380,-12.005
1,Australia,1991,AUS,17353191.0,4.613021e+11,3.183,0.183,279.437,1.376,0.495,...,0.643,1.687,0.007,0.008,0.017,0.002,522.525,339.829,-34.011,-12.171
2,Australia,1992,AUS,17549286.0,4.786337e+11,2.923,0.167,284.433,4.996,1.788,...,0.699,1.680,0.007,0.008,0.017,0.002,568.962,346.607,-36.367,-12.786
3,Australia,1993,AUS,17722905.0,5.015248e+11,3.005,0.170,288.780,4.347,1.528,...,0.810,1.677,0.007,0.008,0.017,0.002,580.666,350.855,-40.004,-13.853
4,Australia,1994,AUS,17897433.0,5.279932e+11,3.484,0.195,293.613,4.833,1.674,...,0.924,1.671,0.007,0.008,0.018,0.002,554.084,353.411,-35.667,-12.148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,United States,2020,USA,339436156.0,1.802736e+13,40.688,0.120,4689.954,-545.958,-10.427,...,30.237,18.090,0.037,0.238,0.286,0.011,5758.456,5309.856,413.131,8.809
346,United States,2021,USA,340161438.0,1.909938e+13,41.312,0.121,5020.111,330.157,7.040,...,31.844,17.980,0.037,0.240,0.288,0.011,6131.129,5675.305,469.552,9.353
347,United States,2022,USA,341534041.0,1.949317e+13,41.884,0.123,5055.403,35.292,0.703,...,30.882,17.870,0.038,0.242,0.291,0.011,6201.504,5728.265,510.586,10.100
348,United States,2023,USA,343477330.0,NaN,40.636,0.118,4918.407,-136.996,-2.710,...,31.847,17.759,0.038,0.245,0.294,0.011,6078.455,5606.243,513.252,10.435


### Creating the Decade Feature

A decade column is created to support decade-wise analysis and visualization.

In [19]:
filtered_dataset["decade"] = ( filtered_dataset["year"] // 10)*10
filtered_dataset[["year","decade"]].head(5)

,year,decade
0,1990,1990
1,1991,1990
2,1992,1990
3,1993,1990
4,1994,1990


### Creating a Time Index

The years_since_1990 feature provides a simple numerical representation of time for regression models.

In [20]:
filtered_dataset['years_since_1990'] = filtered_dataset['year'] - 1990
filtered_dataset[["year","years_since_1990"]].head(5)

,year,years_since_1990
0,1990,0
1,1991,1
2,1992,2
3,1993,3
4,1994,4


### Creating a Five-Year Rolling Mean

A rolling average smooths short-term fluctuations and highlights long-term emission trends.

In [21]:
filtered_dataset['co2_5yr_rolling_mean'] = (

filtered_dataset.groupby('country')['co2']

.transform(lambda x:x.rolling(window=5,min_periods=1).mean())
)

In [22]:
filtered_dataset[["country","year","co2","population","co2_per_capita","co2_5yr_rolling_mean"]].head()

,country,year,co2,population,co2_per_capita,co2_5yr_rolling_mean
0,Australia,1990,278.061,17126304.0,16.236,278.061000
1,Australia,1991,279.437,17353191.0,16.103,278.749000
2,Australia,1992,284.433,17549286.0,16.208,280.643667
3,Australia,1993,288.780,17722905.0,16.294,282.677750
4,Australia,1994,293.613,17897433.0,16.405,284.864800


### Creating Lag Features

Lag features represent emissions from previous years and allow machine learning models to capture temporal relationships between consecutive observations.

In [23]:
filtered_dataset['co2_lag1'] = filtered_dataset.groupby('country')['co2'].shift(1)
filtered_dataset['co2_lag2'] = filtered_dataset.groupby('country')['co2'].shift(2)
filtered_dataset['co2_lag3'] = filtered_dataset.groupby('country')['co2'].shift(3)

In [24]:
filtered_dataset[["country","year","co2","co2_lag1","co2_lag1","co2_lag3"]].head(15)

,country,year,co2,co2_lag1,co2_lag1,co2_lag3
0,Australia,1990,278.061,NaN,NaN,NaN
1,Australia,1991,279.437,278.061,278.061,NaN
2,Australia,1992,284.433,279.437,279.437,NaN
3,Australia,1993,288.780,284.433,284.433,278.061
4,Australia,1994,293.613,288.780,288.780,279.437
5,Australia,1995,304.962,293.613,293.613,284.433
6,Australia,1996,311.851,304.962,304.962,288.780
7,Australia,1997,320.243,311.851,311.851,293.613
8,Australia,1998,334.038,320.243,320.243,304.962
9,Australia,1999,343.876,334.038,334.038,311.851


### Verifying CO₂ Per Capita

The provided CO₂ per capita values are verified by manually calculating them using total CO₂ emissions and population.

In [25]:
filtered_dataset["calculated_co2_per_capita"] = (filtered_dataset["co2"]*1000000 /filtered_dataset["population"])

In [26]:
verification = filtered_dataset[
    filtered_dataset["country"].isin(
        ["India","China","United States"]
    )
][[
    "country",
    "year",
    "co2_per_capita",
    "calculated_co2_per_capita"
]].head(9)

verification

,country,year,co2_per_capita,calculated_co2_per_capita
70,China,1990,2.153,2.152888
71,China,1991,2.237,2.237076
72,China,1992,2.306,2.305715
73,China,1993,2.434,2.434029
74,China,1994,2.559,2.559043
75,China,1995,2.747,2.746581
76,China,1996,2.843,2.842785
77,China,1997,2.830,2.829939
78,China,1998,2.682,2.682451


### Creating Greenhouse Gas Intensity

Greenhouse gas intensity measures total greenhouse gas emissions relative to GDP and provides an indication of emissions efficiency.

In [27]:
filtered_dataset["ghg_intensity"] =(filtered_dataset["total_ghg"] /filtered_dataset["gdp"])

filtered_dataset[[ "country","year","total_ghg","gdp","ghg_intensity"]].head(10)

,country,year,total_ghg,gdp,ghg_intensity
0,Australia,1990,538.479,4.641366e+11,1.160174e-09
1,Australia,1991,522.525,4.613021e+11,1.132718e-09
2,Australia,1992,568.962,4.786337e+11,1.188721e-09
3,Australia,1993,580.666,5.015248e+11,1.157801e-09
4,Australia,1994,554.084,5.279932e+11,1.049415e-09
5,Australia,1995,672.727,5.490387e+11,1.225281e-09
6,Australia,1996,677.191,5.747656e+11,1.178204e-09
7,Australia,1997,649.422,6.014728e+11,1.079720e-09
8,Australia,1998,705.618,6.348471e+11,1.111477e-09
9,Australia,1999,710.665,6.645207e+11,1.069440e-09


### Missing GDP Values

Greenhouse gas intensity cannot be calculated where GDP values are unavailable. These observations remain as missing values.

In [28]:
missing_gdp = filtered_dataset[filtered_dataset["gdp"].isna()]
missing_gdp[["country","year"]].sort_values(["country","year"])

,country,year
33,Australia,2023
34,Australia,2024
68,Brazil,2023
69,Brazil,2024
103,China,2023
104,China,2024
138,Germany,2023
139,Germany,2024
173,India,2023
174,India,2024


### Creating Growth Rate Features

Year-over-year absolute and percentage changes are calculated to quantify annual emission growth.

In [29]:
filtered_dataset["co2_yoy_change"] = (
    filtered_dataset.groupby("country")
    ["co2"]
    .diff()
)
filtered_dataset[["country","year","co2","co2_yoy_change"]].head(15)

,country,year,co2,co2_yoy_change
0,Australia,1990,278.061,NaN
1,Australia,1991,279.437,1.376
2,Australia,1992,284.433,4.996
3,Australia,1993,288.780,4.347
4,Australia,1994,293.613,4.833
5,Australia,1995,304.962,11.349
6,Australia,1996,311.851,6.889
7,Australia,1997,320.243,8.392
8,Australia,1998,334.038,13.795
9,Australia,1999,343.876,9.838


In [30]:
filtered_dataset["co2_yoy_pct_change"] = (
    filtered_dataset.groupby("country")
    ["co2"]
    .pct_change()*100
)
filtered_dataset[["country","year","co2","co2_yoy_change","co2_yoy_pct_change"]].head(15)

,country,year,co2,co2_yoy_change,co2_yoy_pct_change
0,Australia,1990,278.061,NaN,NaN
1,Australia,1991,279.437,1.376,0.494855
2,Australia,1992,284.433,4.996,1.787881
3,Australia,1993,288.780,4.347,1.528304
4,Australia,1994,293.613,4.833,1.673592
5,Australia,1995,304.962,11.349,3.865292
6,Australia,1996,311.851,6.889,2.258970
7,Australia,1997,320.243,8.392,2.691029
8,Australia,1998,334.038,13.795,4.307666
9,Australia,1999,343.876,9.838,2.945174


In [31]:
filtered_dataset["co2_yoy_pct_change"] = (filtered_dataset["co2_yoy_pct_change"].replace([np.inf,-np.inf],np.nan))

### Countries with Highest Average CO₂ Growth

The average annual percentage growth rate is calculated for each country.

In [32]:
growth = (
    filtered_dataset.groupby("country")
    ["co2_yoy_pct_change"].mean().sort_values(
        ascending=False
    ).head(5)
)
growth

country
India           5.207068
China           4.905114
Brazil          2.463847
South Africa    1.132051
Australia       0.990026
Name: co2_yoy_pct_change, dtype: float64

### Countries with Largest Average CO₂ Reduction

The average annual reduction in CO₂ emissions is calculated for each country.

In [33]:
reduction = (
    filtered_dataset.groupby("country")
    ["co2_yoy_change"].mean().sort_values(
        ascending=True
    ).head(5)
)
reduction

country
Russia           -22.228235
Germany          -14.190500
United Kingdom    -8.501147
United States     -6.695324
Japan             -5.676059
Name: co2_yoy_change, dtype: float64

### Creating the Final Feature Dataset

Only the variables required for machine learning and forecasting are retained in the final feature dataset.

In [34]:
feature_cols = [
'country',
'year',
'co2',
'co2_per_capita',
'co2_5yr_rolling_mean',
'co2_lag1',
'co2_lag2',
'co2_lag3',
'co2_yoy_pct_change',
'ghg_intensity'
]
df_features = filtered_dataset[
feature_cols]
df_features.head() 

,country,year,co2,co2_per_capita,co2_5yr_rolling_mean,co2_lag1,co2_lag2,co2_lag3,co2_yoy_pct_change,ghg_intensity
0,Australia,1990,278.061,16.236,278.061000,NaN,NaN,NaN,NaN,1.160174e-09
1,Australia,1991,279.437,16.103,278.749000,278.061,NaN,NaN,0.494855,1.132718e-09
2,Australia,1992,284.433,16.208,280.643667,279.437,278.061,NaN,1.787881,1.188721e-09
3,Australia,1993,288.780,16.294,282.677750,284.433,279.437,278.061,1.528304,1.157801e-09
4,Australia,1994,293.613,16.405,284.864800,288.780,284.433,279.437,1.673592,1.049415e-09


### Inspecting the Final Feature Dataset

The engineered feature dataset is inspected to verify the data types, number of observations, and missing values before exporting it for machine learning.

In [35]:
df_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country               350 non-null    str    
 1   year                  350 non-null    int64  
 2   co2                   350 non-null    float64
 3   co2_per_capita        350 non-null    float64
 4   co2_5yr_rolling_mean  350 non-null    float64
 5   co2_lag1              340 non-null    float64
 6   co2_lag2              330 non-null    float64
 7   co2_lag3              320 non-null    float64
 8   co2_yoy_pct_change    340 non-null    float64
 9   ghg_intensity         330 non-null    float64
dtypes: float64(8), int64(1), str(1)
memory usage: 30.3 KB


In [36]:
df_features.isna().sum()

country                  0
year                     0
co2                      0
co2_per_capita           0
co2_5yr_rolling_mean     0
co2_lag1                10
co2_lag2                20
co2_lag3                30
co2_yoy_pct_change      10
ghg_intensity           20
dtype: int64

### Saving the Feature Dataset

The engineered feature dataset is exported as **ghg_features.csv** for use in Week 3 regression modelling.

In [37]:
df_features.to_csv(
    "data/ghg_features.csv",
    index=False
)

In [38]:
print("Feature Dataset Saved")

Feature Dataset Saved


## Week 2 Summary

Feature engineering was completed by generating time-series features and emission-related indicators. These features provide additional information for predictive modeling and forecasting.